# Part 8 · Notebook 04 — Is the Sharpe real? Significance and capacity

**Sessions:** S7 (Statistical significance) · S8 (Capacity & backtest-vs-paper reconciliation) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Put an error bar on a Sharpe ratio, analytically and by bootstrap.
2. Compute the Probabilistic Sharpe Ratio and the minimum track record length.
3. Find the capital at which impact costs kill a strategy.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

In [ ]:
strats = p.strategy_returns()
true_sr = dict(zip(strats.columns, (0.8, 0.6, 0.5, 0.4, 0.3)))
pd.DataFrame({"true Sharpe": true_sr, "estimated (10 years)": {s: p.sharpe(strats[s]) for s in strats}}).round(2)

## 1. The Sharpe ratio's error bar

With i.i.d. returns the standard error of a per-period Sharpe `SR` over `T` periods is `√((1 + SR²/2) / T)` (Lo, 2002). In annual units multiply by √252. For ten years of a 0.8 Sharpe strategy that is about ±0.32: a 95% interval from roughly 0.2 to 1.4.

In [ ]:
r = strats["S1"].to_numpy()
sr_d = r.mean() / r.std(ddof=1)
se = p.sharpe_se(sr_d, len(r)) * np.sqrt(252)
boot = p.stationary_bootstrap_sharpes(r, n_boot=400) * np.sqrt(252)
print(f"S1: Sharpe {sr_d * np.sqrt(252):.2f} ± {se:.2f} (analytic);  bootstrap 95% interval {np.percentile(boot, 2.5):.2f} to {np.percentile(boot, 97.5):.2f}")
plt.hist(boot, bins=30); plt.axvline(0.8, color="black", lw=1, label="true 0.8"); plt.legend(); plt.title("Stationary bootstrap of S1's Sharpe"); plt.show()

## 2. The Probabilistic Sharpe Ratio

PSR is the probability that the true Sharpe exceeds a benchmark `SR*`, correcting for the sample length **and** for skew and fat tails: `Φ((SR − SR*)·√(T − 1) / √(1 − γ3·SR + (γ4 − 1)/4·SR²))`, with per-period `SR` (ddof=1), `γ3` the skewness and `γ4` the **non-excess** kurtosis (`scipy.stats.kurtosis(r, fisher=False)`, 3 for a normal).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from scipy.stats import norm, skew, kurtosis

def psr(returns, sr_benchmark=0.0):
    r = np.asarray(returns, dtype=float)
    T = r.size
    sr = r.mean() / r.std(ddof=1)
    g3, g4 = skew(r), kurtosis(r, fisher=False)
    return ...                                    # ✍️ Φ(...)

mine = [p.attempt(psr, strats[s]) for s in strats]
mine = p.check("psr", mine, [p.psr(strats[s]) for s in strats])
pd.Series(mine, index=strats.columns, name="P(true Sharpe > 0)").round(4)

Turn it around: how long must a track record be before a Sharpe this size is significant at 95%? `MinTRL = 1 + (1 − γ3·SR + (γ4 − 1)/4·SR²) · (z_{0.95} / (SR − SR*))²` periods (infinite if `SR <= SR*`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def min_track_record(returns, sr_benchmark=0.0, alpha=0.05):
    r = np.asarray(returns, dtype=float)
    sr = r.mean() / r.std(ddof=1)
    if sr <= sr_benchmark:
        return float("inf")
    g3, g4 = skew(r), kurtosis(r, fisher=False)
    return ...                                    # ✍️ in periods (days)

mine = [p.attempt(min_track_record, strats[s]) for s in strats]
mine = p.check("min_track_record", mine, [p.min_track_record(strats[s]) for s in strats])
pd.Series(np.array(mine) / 252, index=strats.columns, name="years needed at 95%").round(1)

## 3. Capacity

A strategy's edge is fixed; its costs grow with size. With turnover `τ` (fraction of capital traded per day) and the square-root law, the daily cost is `τ · k · σ_daily · √(τ · C / ADV)` for capital `C`. `p.net_sharpe_vs_capital` gives the net Sharpe for a list of capitals. **Capacity** is the largest capital whose net Sharpe is still at least `min_sharpe` (0 if none).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def capacity(sharpe_by_capital, min_sharpe=0.5):
    ok = ...                                      # ✍️ the entries at or above min_sharpe
    return float(ok.index.max()) if len(ok) else 0.0

caps = [1e6, 3e6, 1e7, 3e7, 1e8, 3e8, 1e9]
curves = {name: p.net_sharpe_vs_capital(strats["S1"], turnover, adv_dollars=5e8, daily_vol=0.015, capitals=caps)
          for name, turnover in [("slow (5% a day)", 0.05), ("fast (100% a day)", 1.0)]}
mine = [p.attempt(capacity, cv) for cv in curves.values()]
mine = p.check("capacity", mine, [p.capacity(cv) for cv in curves.values()])
fig, ax = plt.subplots()
for (name, cv), cap in zip(curves.items(), mine):
    ax.semilogx(cv.index, cv.values, "o-", label=f"{name}: capacity ${cap:,.0f}")
ax.axhline(0.5, color=p.PALETTE[7], ls="--", lw=1); ax.set(xlabel="capital, $", ylabel="net Sharpe"); ax.legend(); plt.show()

Same gross returns, same market: turnover alone decides how much money the idea can take.

## Wrap-up

* Report every Sharpe with an interval; PSR corrects for sample length, skew and tails.
* Low-Sharpe strategies need decades to prove themselves: combine evidence, don't wait for significance alone.
* Capacity comes from turnover and impact, not from the idea.
* Graded version: `labs/part08/week26_analysis` (trade reshuffling, implementation shortfall).